# African Motor Claims: dataset exploration

This notebook inspects the dataset before any model is trained. It checks the target balance, missing values, selected features and chronological split.

> **Research boundary:** the dataset is hosted on Hugging Face, but the records and fraud labels are synthetic. Results are useful for developing the pipeline, not for claiming real-world fraud performance.

## 1. Set up the project

The same cell works in Colab and when Jupyter is started from this repository.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

# Colab starts with an empty workspace, while local Jupyter already has the project.
IN_COLAB = "COLAB_RELEASE_TAG" in os.environ
REPOSITORY_URL = "https://github.com/ganesh1997oli/Decentralized-Claims-Registry"

if IN_COLAB:
    # Clone the repository once so the notebook can reuse the project code.
    PROJECT_ROOT = Path("/content/Decentralized-Claims-Registry")
    if not PROJECT_ROOT.exists():
        subprocess.run(["git", "clone", REPOSITORY_URL, str(PROJECT_ROOT)], check=True)
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", str(PROJECT_ROOT / "model/requirements.txt")],
        check=True,
    )
    os.chdir(PROJECT_ROOT)
else:
    # When running locally, walk upwards until we find the repository root.
    current = Path.cwd().resolve()
    PROJECT_ROOT = next(
        (path for path in (current, *current.parents) if (path / "model/research_pipeline.py").exists()),
        None,
    )
    if PROJECT_ROOT is None:
        raise RuntimeError("Start Jupyter from inside the repository.")
    os.chdir(PROJECT_ROOT)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project: {PROJECT_ROOT}")
print(f"Running in Colab: {IN_COLAB}")

## 2. Download the reviewed dataset revision

The project pins the Hugging Face revision and verifies its SHA-256 checksum. Existing data is reused.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from model.download_dataset import (
    DATASET_REPOSITORY,
    DATASET_REVISION,
    DEFAULT_DATASET_PATH,
    download_dataset,
    file_sha256,
)

# Reuse the local copy when it exists. Otherwise download the exact reviewed version.
dataset_path = DEFAULT_DATASET_PATH
if not dataset_path.exists():
    download_dataset(dataset_path)

raw_claims = pd.read_csv(dataset_path)
print(f"Dataset: {DATASET_REPOSITORY}@{DATASET_REVISION}")
print(f"SHA-256: {file_sha256(dataset_path)}")
print(f"Rows: {len(raw_claims):,} | Columns: {len(raw_claims.columns)}")
raw_claims.head()

## 3. Check structure and data quality

In [ ]:
# This compact table helps us spot missing data or suspiciously constant columns.
quality = pd.DataFrame(
    {
        "dtype": raw_claims.dtypes.astype(str),
        "missing": raw_claims.isna().sum(),
        "unique": raw_claims.nunique(dropna=False),
    }
)
quality

In [ ]:
# Claim IDs should be unique and the target should contain only 0 or 1.
duplicate_ids = int(raw_claims["claim_id"].duplicated().sum())
invalid_targets = int((~raw_claims["fraud_flag"].isin([0, 1])).sum())
print(f"Duplicate claim IDs: {duplicate_ids}")
print(f"Invalid target values: {invalid_targets}")

## 4. Inspect fraud-label balance

Fraud is the minority class, so accuracy alone would be misleading. The training notebook therefore reports precision, recall, F1 and PR-AUC.

In [ ]:
# Show both the raw counts and percentages so the imbalance is easy to understand.
target_counts = raw_claims["fraud_flag"].value_counts().sort_index()
target_summary = pd.DataFrame(
    {
        "label": ["Not fraud", "Fraud"],
        "count": target_counts.reindex([0, 1], fill_value=0).values,
        "percentage": (target_counts.reindex([0, 1], fill_value=0).values / len(raw_claims) * 100).round(2),
    },
    index=[0, 1],
)
display(target_summary)

ax = target_summary.set_index("label")["count"].plot(
    kind="bar", color=["#4C78A8", "#E45756"], rot=0, title="Synthetic fraud-label balance"
)
ax.set_ylabel("Claims")
plt.tight_layout()
plt.show()

## 5. Explore a few useful relationships

These charts describe the generated dataset. They should not be interpreted as real country or market fraud rates.

In [ ]:
# These two simple views give us a feel for amount skew and generated label patterns.
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
raw_claims["claim_amount_usd"].plot(
    kind="hist", bins=40, ax=axes[0], color="#4C78A8", title="Claim amount distribution"
)
axes[0].set_xlabel("Claim amount (USD)")

raw_claims.groupby("claim_type")["fraud_flag"].mean().sort_values().plot(
    kind="barh", ax=axes[1], color="#F58518", title="Generated fraud rate by claim type"
)
axes[1].set_xlabel("Fraud-label rate")
plt.tight_layout()
plt.show()

In [ ]:
# These are properties of the synthetic generator, not measured national fraud rates.
country_summary = (
    raw_claims.groupby("country")
    .agg(claims=("claim_id", "size"), fraud_rate=("fraud_flag", "mean"))
    .sort_values("fraud_rate", ascending=False)
)
country_summary.style.format({"claims": "{:,}", "fraud_rate": "{:.2%}"})

## 6. Keep only submission-time features

The reusable pipeline removes fields that happen after investigation or reveal how the synthetic target was generated.

In [ ]:
from model.research_pipeline import (
    FEATURE_COLUMNS,
    LEAKAGE_COLUMNS,
    TARGET_COLUMN,
    prepare_claims,
    temporal_split,
)

# Put included and excluded fields side by side to make the leakage decision visible.
feature_decisions = pd.DataFrame(
    {
        "used_by_model": pd.Series(FEATURE_COLUMNS),
        "excluded_to_avoid_leakage": pd.Series(LEAKAGE_COLUMNS),
    }
)
feature_decisions

## 7. Verify the chronological split

Earlier claims train the models. Later claims are reserved for validation and final testing, which is closer to how a deployed model encounters future claims.

In [ ]:
# Sort by claim date before splitting so the model is tested on later claims.
claims = prepare_claims(raw_claims)
split = temporal_split(claims)

split_summary = pd.DataFrame(
    [
        {
            "split": name,
            "rows": len(part),
            "start_date": part["claim_date"].min().date(),
            "end_date": part["claim_date"].max().date(),
            "fraud_rate": part[TARGET_COLUMN].mean(),
        }
        for name, part in [("Train", split.train), ("Validation", split.validation), ("Test", split.test)]
    ]
)
split_summary.style.format({"rows": "{:,}", "fraud_rate": "{:.2%}"})

## Findings to record

- The file is reproducible because the revision and checksum are pinned.
- The target is imbalanced, so fraud-focused metrics are required.
- Post-outcome and target-generation fields are excluded to reduce leakage.
- The final test period remains untouched until model evaluation.
- All observations and labels are synthetic, so external validation would still be required.